# 02 · Web scraping SBS (TAMN y TIPMN diarias)
Ejecutar en la misma sesión de Colab, después del 01.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Nombres y apellidos: ASTETE ROSALES GRAYCE NICOL
# Código de matrícula: 2024200482A
# Tema N.º 2 del temario: El margen de intermediación financiera en el Perú: spread TAMN-TIPMN y sus determinantes
# Fecha de extracción:2026-09-25
# ============================================================================
# 02_scraping_web | Vía 2: web scraping del portal de la SBS
# TAMN y TIPMN DIARIAS (BCRPData solo las tiene mensuales, por eso no van por API)
# CELDA 1. Librerías y parámetros
# ============================================================================
import re, time, gzip, logging
from io import StringIO
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

CODIGO_MATRICULA = "2024200482A"
PAGINAS = {   # columna : (URL del portal SBS, etiqueta en la tabla, descripción, unidad)
    "tamn_sbs_pct":  ("https://www.sbs.gob.pe/app/pp/EstadisticasSAEEPortal/Paginas/TIActivaMercado.aspx?tip=B",
                      "TAMN", "Tasa activa promedio de mercado en moneda nacional", "% efectivo anual"),
    "tipmn_sbs_pct": ("https://www.sbs.gob.pe/app/pp/EstadisticasSAEEPortal/Paginas/TIPasivaMercado.aspx?tip=B",
                      "TIPMN", "Tasa pasiva promedio de mercado en moneda nacional", "% efectivo anual"),
}
PAUSA = 1.5                      # >= 1 s entre solicitudes
FORMATO_FECHA = "%d/%m/%Y"       # confirmar en la CELDA 3
GUARDAR_HTML = True              # cada página recibida se guarda comprimida como evidencia
CABECERAS = {"User-Agent": f"UNCP-Finanzas-I-055D/1.0 (investigacion academica; e_{CODIGO_MATRICULA}@uncp.edu.pe)"}

RAIZ = Path("/content/drive/MyDrive/Base de datos y código Grayce Nicol Astete Rosales")
DIR_HTML = RAIZ / "datos_crudos" / "html_sbs"
DIR_HTML.mkdir(parents=True, exist_ok=True)
ARCHIVO_AVANCE = RAIZ / "datos_crudos" / "avance_scraping_sbs.csv"
# Se consultan exactamente los mismos días hábiles que devolvió la API (evita feriados)
FECHAS = pd.read_csv(RAIZ / "datos_crudos" / f"datos_crudos_api_{CODIGO_MATRICULA}.csv", sep=";")["fecha"].tolist()

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S",
                    handlers=[logging.FileHandler(RAIZ / "log_ejecucion.txt", encoding="utf-8"),
                              logging.StreamHandler()], force=True)
log = logging.getLogger()
sesion = requests.Session()
sesion.headers.update(CABECERAS)
print(f"Días a consultar: {len(FECHAS)} ({FECHAS[0]} a {FECHAS[-1]})")

Días a consultar: 1304 (2021-01-01 a 2025-12-31)


In [ ]:
# ============================================================================
# CELDA 2. Revisión del robots.txt (numeral 2.4.8). Si prohíbe la ruta -> numeral 2.4.3
# ============================================================================
RUTA_OBJETIVO = "/app/pp/EstadisticasSAEEPortal/"
r = sesion.get("https://www.sbs.gob.pe/robots.txt", timeout=30)
log.info(f"robots.txt SBS | HTTP {r.status_code}")
print(r.text[:1500])
prohibidas = [l.split(":", 1)[1].strip() for l in r.text.splitlines()
              if l.lower().startswith("disallow:") and l.split(":", 1)[1].strip()]
bloqueada = any(RUTA_OBJETIVO.startswith(p) for p in prohibidas)
log.info(f"¿robots.txt prohíbe {RUTA_OBJETIVO}?: {bloqueada}")

2026-09-25 01:23:31 | robots.txt SBS | HTTP 200
2026-09-25 01:23:31 | ¿robots.txt prohíbe /app/pp/EstadisticasSAEEPortal/?: False


# Begin robots.txt file
#/-----------------------------------------------\
#| In single portal/domain situations, uncomment the sitmap line and enter domain name
#\-----------------------------------------------/
#Sitemap: http://www.DomainNamehere.com/sitemap.aspx


User-agent: *
Disallow: /admin/
Disallow: /App_Browsers/
Disallow: /App_Code/
Disallow: /App_Data/
Disallow: /App_GlobalResources/
Disallow: /bin/
Disallow: /Components/
Disallow: /Config/
Disallow: /contest/
Disallow: /controls/
Disallow: /DesktopModules/
Disallow: /Documentation/
Disallow: /HttpModules/
Disallow: /images/
Disallow: /Install/
Disallow: /js/
Disallow: /Portals/
Disallow: /Providers/
Disallow: /Resources/ContentRotator/
Disallow: /Resources/ControlPanel/
Disallow: /Resources/Dashboard/
Disallow: /Resources/FeedBrowser/
Disallow: /Resources/OpenForceAd/
Disallow: /Resources/Search/
Disallow: /Resources/Shared/
Disallow: /Resources/SkinWidgets/
Disallow: /Resources/TabStrip/
Disallow: /Resources/Widgets/
Disa

In [ ]:
# ============================================================================
# CELDA 3. Diagnóstico del formulario (EJECUTAR Y LEER ANTES DE LA CELDA 5)
# Las páginas de la SBS son ASP.NET: guardan su estado en campos ocultos
# (__VIEWSTATE, __EVENTVALIDATION) y la fecha se envía por POST.
# Esta celda muestra los campos para confirmar el nombre del campo de fecha y del botón.
# ============================================================================
def campos_formulario(soup):
    """Devuelve los campos del primer formulario: nombre, tipo y valor."""
    form = soup.find("form") or soup
    campos = []
    for inp in form.find_all("input"):
        campos.append({"tag": "input", "name": inp.get("name"), "id": inp.get("id"),
                       "type": (inp.get("type") or "text").lower(), "value": (inp.get("value") or "")[:40]})
    for sel in form.find_all("select"):
        op = sel.find("option", selected=True) or sel.find("option")
        campos.append({"tag": "select", "name": sel.get("name"), "id": sel.get("id"),
                       "type": "select", "value": op.get("value") if op else ""})
    return pd.DataFrame(campos)

for var, (url, etiqueta, *_ ) in PAGINAS.items():
    r = sesion.get(url, timeout=60)
    log.info(f"Diagnóstico {var} | GET {url} | HTTP {r.status_code} | {len(r.content)} bytes")
    soup = BeautifulSoup(r.text, "lxml")
    print(f"\n===== {var.upper()} =====  ¿La página contiene '{etiqueta}'?: {etiqueta in r.text}")
    display(campos_formulario(soup))

2026-09-25 01:24:19 | Diagnóstico tamn_sbs_pct | GET https://www.sbs.gob.pe/app/pp/EstadisticasSAEEPortal/Paginas/TIActivaMercado.aspx?tip=B | HTTP 200 | 58558 bytes



===== TAMN_SBS_PCT =====  ¿La página contiene 'TAMN'?: True


,tag,name,id,type,value
0,input,ctl00_MainScriptManager_TSM,ctl00_MainScriptManager_TSM,hidden,
1,input,__VIEWSTATE,__VIEWSTATE,hidden,/wEPDwUJNjEwMzE3MDg5D2QWAmYPZBYCAgMPZBYC
2,input,__VIEWSTATEGENERATOR,__VIEWSTATEGENERATOR,hidden,E2E416CF
3,input,__EVENTVALIDATION,__EVENTVALIDATION,hidden,/wEdAARZ0OBeAz76lM3Hkp0erHPnlawXGEyscjwH
4,input,ctl00$cphContent$rdpDate,ctl00_cphContent_rdpDate,text,2026-09-24
5,input,ctl00$cphContent$rdpDate$dateInput,ctl00_cphContent_rdpDate_dateInput,text,24/09/2026
6,input,ctl00_cphContent_rdpDate_dateInput_ClientState,ctl00_cphContent_rdpDate_dateInput_ClientState,hidden,
7,input,ctl00_cphContent_rdpDate_calendar_SD,ctl00_cphContent_rdpDate_calendar_SD,hidden,[]
8,input,ctl00_cphContent_rdpDate_calendar_AD,ctl00_cphContent_rdpDate_calendar_AD,hidden,"[[1000,1,1],[2099,12,30],[2026,9,24]]"
9,input,ctl00_cphContent_rdpDate_ClientState,ctl00_cphContent_rdpDate_ClientState,hidden,


2026-09-25 01:24:19 | Diagnóstico tipmn_sbs_pct | GET https://www.sbs.gob.pe/app/pp/EstadisticasSAEEPortal/Paginas/TIPasivaMercado.aspx?tip=B | HTTP 200 | 51311 bytes



===== TIPMN_SBS_PCT =====  ¿La página contiene 'TIPMN'?: True


,tag,name,id,type,value
0,input,ctl00_MainScriptManager_TSM,ctl00_MainScriptManager_TSM,hidden,
1,input,__VIEWSTATE,__VIEWSTATE,hidden,/wEPDwUKMTUwNTEyNTUwMQ9kFgJmD2QWAgIDD2QW
2,input,__VIEWSTATEGENERATOR,__VIEWSTATEGENERATOR,hidden,A077754B
3,input,__EVENTVALIDATION,__EVENTVALIDATION,hidden,/wEdAARnVxTbhw8MVKQrXY0TpiVVlawXGEyscjwH
4,input,ctl00$cphContent$rdpDate,ctl00_cphContent_rdpDate,text,2026-09-24
5,input,ctl00$cphContent$rdpDate$dateInput,ctl00_cphContent_rdpDate_dateInput,text,24/09/2026
6,input,ctl00_cphContent_rdpDate_dateInput_ClientState,ctl00_cphContent_rdpDate_dateInput_ClientState,hidden,
7,input,ctl00_cphContent_rdpDate_calendar_SD,ctl00_cphContent_rdpDate_calendar_SD,hidden,[]
8,input,ctl00_cphContent_rdpDate_calendar_AD,ctl00_cphContent_rdpDate_calendar_AD,hidden,"[[1000,1,1],[2099,12,30],[2026,9,24]]"
9,input,ctl00_cphContent_rdpDate_ClientState,ctl00_cphContent_rdpDate_ClientState,hidden,


In [ ]:
# ============================================================================
# CELDA 4. Funciones de scraping (ajustadas al formulario real de la SBS)
# El selector de fecha es un control Telerik RadDatePicker: la fecha se envía
# en 4 campos a la vez, cada uno con su propio formato.
# ============================================================================
import json

CAMPO_FECHA_ISO   = "ctl00$cphContent$rdpDate"                      # AAAA-MM-DD
CAMPO_FECHA_TEXTO = "ctl00$cphContent$rdpDate$dateInput"            # DD/MM/AAAA
CAMPO_ESTADO      = "ctl00_cphContent_rdpDate_dateInput_ClientState" # estado JSON del control
CAMPO_CALENDARIO  = "ctl00_cphContent_rdpDate_calendar_SD"          # fecha seleccionada
CAMPO_BOTON       = "ctl00$cphContent$btnConsultar"                 # botón "Consultar"

def armar_payload(soup, fecha):
    """Copia todos los campos del formulario (incluido __VIEWSTATE), coloca la fecha
    en los 4 campos del calendario y agrega solo el botón Consultar."""
    form = soup.find("form") or soup
    datos = {}
    for inp in form.find_all("input"):
        nombre, tipo = inp.get("name"), (inp.get("type") or "text").lower()
        if not nombre or tipo in ("submit", "button", "image"):
            continue                                   # los botones no se copian
        if tipo in ("checkbox", "radio") and not inp.has_attr("checked"):
            continue
        datos[nombre] = inp.get("value", "")
    for sel in form.find_all("select"):
        op = sel.find("option", selected=True) or sel.find("option")
        if sel.get("name") and op is not None:
            datos[sel["name"]] = op.get("value", "")

    iso, texto = fecha.strftime("%Y-%m-%d"), fecha.strftime("%d/%m/%Y")
    telerik = fecha.strftime("%Y-%m-%d-00-00-00")
    datos[CAMPO_FECHA_ISO] = iso
    datos[CAMPO_FECHA_TEXTO] = texto
    datos[CAMPO_ESTADO] = json.dumps({"enabled": True, "emptyMessage": "", "validationText": telerik,
                                      "valueAsString": telerik, "minDateStr": "1000-01-01-00-00-00",
                                      "maxDateStr": "2099-12-31-00-00-00", "lastSetTextBoxValue": texto},
                                     separators=(",", ":"))
    datos[CAMPO_CALENDARIO] = f"[[{fecha.year},{fecha.month},{fecha.day}]]"
    datos[CAMPO_BOTON] = "Consultar"
    datos["__EVENTTARGET"] = ""
    datos["__EVENTARGUMENT"] = ""
    return datos

def extraer_valor(html, etiqueta):
    """Busca la etiqueta (TAMN o TIPMN) en las tablas y devuelve el primer número
    de esa fila como texto original."""
    try:
        tablas = pd.read_html(StringIO(html), flavor="lxml")
    except ValueError:
        tablas = []
    for t in tablas:
        for _, fila in t.astype(str).iterrows():
            celdas = [c.strip() for c in fila.tolist()]
            for i, c in enumerate(celdas):
                if re.fullmatch(rf"{etiqueta}\b.*", c, re.I):
                    for siguiente in celdas[i + 1:]:
                        if re.fullmatch(r"-?\d+[.,]?\d*", siguiente):
                            return siguiente
    m = re.search(rf"{etiqueta}[^\d]{{0,60}}?(\d+[.,]\d+)", BeautifulSoup(html, "lxml").get_text(" "))
    return m.group(1) if m else None

def fecha_en_pagina(soup):
    """Fecha que quedó cargada en el calendario tras la consulta."""
    campo = soup.find("input", attrs={"name": CAMPO_FECHA_TEXTO})
    return campo.get("value") if campo else None

def consultar_dia(url, etiqueta, fecha, soup_previa):
    """POST del formulario para una fecha. Devuelve (valor_texto, fecha_reportada, http, soup_nueva)."""
    r = sesion.post(url, data=armar_payload(soup_previa, fecha), timeout=60)
    if GUARDAR_HTML:
        with gzip.open(DIR_HTML / f"{etiqueta}_{fecha:%Y%m%d}.html.gz", "wb") as f:
            f.write(r.content)
    soup = BeautifulSoup(r.text, "lxml")
    return extraer_valor(r.text, etiqueta), fecha_en_pagina(soup), r.status_code, soup

In [ ]:
# ============================================================================
# CELDA 5. Prueba con 3 fechas + diagnóstico de dónde aparece la etiqueta
# ============================================================================
for var, (url, etiqueta, *_ ) in PAGINAS.items():
    soup = BeautifulSoup(sesion.get(url, timeout=60).text, "lxml")
    for f in pd.to_datetime(["2025-12-01", "2025-12-02", "2025-12-03"]):
        valor, freportada, http, soup = consultar_dia(url, etiqueta, f, soup)
        print(f"{etiqueta} pedida {f:%d/%m/%Y} -> valor {valor} | fecha en página {freportada} | HTTP {http}")
        time.sleep(PAUSA)
    # Texto alrededor de la etiqueta en la última respuesta (para revisar si hace falta)
    texto = soup.get_text(" ", strip=True)
    for m in list(re.finditer(etiqueta, texto))[:3]:
        print("   ...", texto[max(0, m.start() - 80): m.end() + 120], "...")

TAMN pedida 01/12/2025 -> valor 15.85 | fecha en página 01/12/2025 | HTTP 200
TAMN pedida 02/12/2025 -> valor 15.94 | fecha en página 02/12/2025 | HTTP 200
TAMN pedida 03/12/2025 -> valor 15.92 | fecha en página 03/12/2025 | HTTP 200
   ... sa de Interés Activa Promedio de Mercado Efectiva al 03/12/2025 Moneda Nacional(TAMN) 15.92 % Anual Factor Diario 0.00041 *Factor Acumulado 1 10,633.78872 Moneda Nacional(TAMN + 1) 16.92 % Anual Factor Di ...
   ...  % Anual Factor Diario 0.00041 *Factor Acumulado 1 10,633.78872 Moneda Nacional(TAMN + 1) 16.92 % Anual Factor Diario 0.00043 *Factor Acumulado 1 21,037.85101 Moneda Nacional(TAMN + 2) 17.92 % Anual Facto ...
   ...  % Anual Factor Diario 0.00043 *Factor Acumulado 1 21,037.85101 Moneda Nacional(TAMN + 2) 17.92 % Anual Factor Diario 0.00046 *Factor Acumulado 1 41,368.49651 Moneda Extranjera(TAMEX) 9.98 % Anual Factor  ...
TIPMN pedida 01/12/2025 -> valor 2.06 | fecha en página 01/12/2025 | HTTP 200
TIPMN pedida 02/12/2025 -> valor 1.94 | 

In [ ]:
# ============================================================================
# CELDA 6. Extracción completa (guarda avance; si Colab se corta, vuelva a ejecutar)
# Tiempo aproximado: 2 páginas x ~1 240 días x ~2,5 s ≈ 1,5 a 2 horas
# ============================================================================
hechos = set()
if ARCHIVO_AVANCE.exists():
    prev = pd.read_csv(ARCHIVO_AVANCE, dtype=str)
    hechos = set(zip(prev["columna"], prev["fecha"]))

for col, (url, etiqueta, *_ ) in PAGINAS.items():
    soup = BeautifulSoup(sesion.get(url, timeout=60).text, "lxml")
    pendientes = [f for f in FECHAS if (col, f) not in hechos]
    log.info(f"{etiqueta}: {len(pendientes)} días pendientes")
    for i, f in enumerate(pendientes, 1):
        try:
            valor, freportada, http, soup = consultar_dia(url, etiqueta, pd.Timestamp(f), soup)
        except Exception as e:
            log.info(f"{etiqueta} {f}: error {e}; se reinicia la sesión")
            time.sleep(10); soup = BeautifulSoup(sesion.get(url, timeout=60).text, "lxml"); continue
        pd.DataFrame([{"columna": col, "fecha": f, "valor": valor, "fecha_en_pagina": freportada, "http": http}]
                     ).to_csv(ARCHIVO_AVANCE, mode="a", header=not ARCHIVO_AVANCE.exists(), index=False)
        if i % 50 == 0 or valor is None:
            log.info(f"{etiqueta} {f} -> {valor} | HTTP {http} | {i}/{len(pendientes)}")
        time.sleep(PAUSA)
log.info("FIN scraping SBS")

2026-09-25 01:32:04 | TAMN: 1304 días pendientes
2026-09-25 01:33:46 | TAMN 2021-03-11 -> 11.08 | HTTP 200 | 50/1304
2026-09-25 01:35:33 | TAMN 2021-05-20 -> 10.73 | HTTP 200 | 100/1304
2026-09-25 01:37:18 | TAMN 2021-07-29 -> 10.66 | HTTP 200 | 150/1304
2026-09-25 01:39:06 | TAMN 2021-10-07 -> 10.42 | HTTP 200 | 200/1304
2026-09-25 01:40:42 | TAMN 2021-12-08 -> None | HTTP 200 | 244/1304
2026-09-25 01:40:55 | TAMN 2021-12-16 -> 10.93 | HTTP 200 | 250/1304
2026-09-25 01:42:41 | TAMN 2022-02-24 -> 11.34 | HTTP 200 | 300/1304
2026-09-25 01:44:29 | TAMN 2022-05-05 -> 12.02 | HTTP 200 | 350/1304
2026-09-25 01:46:17 | TAMN 2022-07-14 -> 12.68 | HTTP 200 | 400/1304
2026-09-25 01:48:06 | TAMN 2022-09-22 -> 13.38 | HTTP 200 | 450/1304
2026-09-25 01:49:55 | TAMN 2022-12-01 -> 14.19 | HTTP 200 | 500/1304
2026-09-25 01:51:38 | TAMN 2023-02-09 -> 14.66 | HTTP 200 | 550/1304
2026-09-25 01:53:21 | TAMN 2023-04-20 -> 15.17 | HTTP 200 | 600/1304
2026-09-25 01:55:11 | TAMN 2023-06-29 -> 15.68 | HTTP 20

In [ ]:
print("ARCHIVO_AVANCE =", ARCHIVO_AVANCE)
print("¿Existe? =", ARCHIVO_AVANCE.exists())

if ARCHIVO_AVANCE.exists():
    print("Tamaño:", ARCHIVO_AVANCE.stat().st_size, "bytes")

ARCHIVO_AVANCE = /content/drive/MyDrive/Base de datos y código Grayce Nicol Astete Rosales/datos_crudos/avance_scraping_sbs.csv
¿Existe? = True
Tamaño: 117396 bytes


In [ ]:
# ============================================================================
# CELDA 7. TABLA DE DATOS CRUDOS SBS: id | fecha | tamn_sbs_pct | tipmn_sbs_pct
# ============================================================================
avance = pd.read_csv(ARCHIVO_AVANCE, dtype=str).drop_duplicates(["columna", "fecha"], keep="last")
avance["valor"] = pd.to_numeric(avance["valor"].str.replace(",", "."), errors="coerce")
tabla_sbs = (avance.pivot(index="fecha", columns="columna", values="valor")
             .reindex(FECHAS)[list(PAGINAS)].rename_axis("fecha").reset_index())
tabla_sbs.columns.name = None
tabla_sbs.insert(0, "id", range(1, len(tabla_sbs) + 1))
archivo = RAIZ / "datos_crudos" / f"datos_crudos_sbs_{CODIGO_MATRICULA}"
tabla_sbs.to_csv(f"{archivo}.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
tabla_sbs.to_excel(f"{archivo}.xlsx", index=False)
display(tabla_sbs.head(10))

,id,fecha,tamn_sbs_pct,tipmn_sbs_pct
0,1,2021-01-01,12.10,0.98
1,2,2021-01-04,12.23,0.98
2,3,2021-01-05,12.15,0.97
3,4,2021-01-06,12.10,0.96
4,5,2021-01-07,12.08,0.96
5,6,2021-01-08,12.05,0.96
6,7,2021-01-11,12.06,0.96
7,8,2021-01-12,12.11,0.95
8,9,2021-01-13,12.10,0.95
9,10,2021-01-14,12.10,0.95


In [ ]:
# ============================================================================
# CELDA 8. TABLA ÚNICA DE DATOS CRUDOS (SBS + API), unida por la llave 'fecha'
# Hoja 1: datos_crudos | Hoja 2: diccionario de variables
# ============================================================================
api = pd.read_csv(RAIZ / "datos_crudos" / f"datos_crudos_api_{CODIGO_MATRICULA}.csv", sep=";", decimal=",")
tabla = tabla_sbs.drop(columns="id").merge(api.drop(columns="id"), on="fecha", how="outer").sort_values("fecha")
tabla.insert(0, "id", range(1, len(tabla) + 1))

API_INFO = {  # misma información del cuaderno 01
    "tasa_referencia_bcrp_pct":  ("BCRPData (API)", "PD12301MD", "Tasa de referencia de la política monetaria del BCRP", "% anual"),
    "tasa_interbancaria_mn_pct": ("BCRPData (API)", "PD04692MD", "Tasa de interés interbancaria en moneda nacional", "% anual"),
    "riesgo_pais_embig_pbs":     ("BCRPData (API)", "PD04709XD", "Spread EMBIG Perú (riesgo país)", "puntos básicos"),
    "tipo_cambio_venta_pen_usd": ("BCRPData (API)", "PD04638PD", "Tipo de cambio interbancario, venta", "S/ por US$"),
}
filas = [{"variable": "id", "descripcion": "Número de fila", "unidad": "-", "fuente": "-", "codigo_o_url": "-"},
         {"variable": "fecha", "descripcion": "Día hábil (llave de unión)", "unidad": "AAAA-MM-DD", "fuente": "-", "codigo_o_url": "-"}]
filas += [{"variable": c, "descripcion": d, "unidad": u, "fuente": "SBS (web scraping)", "codigo_o_url": url}
          for c, (url, _, d, u) in PAGINAS.items()]
filas += [{"variable": c, "descripcion": d, "unidad": u, "fuente": f, "codigo_o_url": cod}
          for c, (f, cod, d, u) in API_INFO.items()]
diccionario = pd.DataFrame(filas)
diccionario["observaciones"] = [len(tabla), len(tabla)] + [pd.to_numeric(tabla[c], errors="coerce").notna().sum()
                                                            for c in diccionario["variable"][2:]]

archivo = RAIZ / "datos_crudos" / f"datos_crudos_{CODIGO_MATRICULA}"
tabla.to_csv(f"{archivo}.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
with pd.ExcelWriter(f"{archivo}.xlsx") as xls:
    tabla.to_excel(xls, sheet_name="datos_crudos", index=False)
    diccionario.to_excel(xls, sheet_name="diccionario", index=False)
display(diccionario)
display(tabla.head(10))

import shutil
shutil.make_archive("carpeta3_datos_crudos", "zip", RAIZ)
try:
    from google.colab import files; files.download("carpeta3_datos_crudos.zip")
except ImportError:
    print("Zip creado: carpeta3_datos_crudos.zip")

,variable,descripcion,unidad,fuente,codigo_o_url,observaciones
0,id,Número de fila,-,-,-,1304
1,fecha,Día hábil (llave de unión),AAAA-MM-DD,-,-,1304
2,tamn_sbs_pct,Tasa activa promedio de mercado en moneda naci...,% efectivo anual,SBS (web scraping),https://www.sbs.gob.pe/app/pp/EstadisticasSAEE...,1303
3,tipmn_sbs_pct,Tasa pasiva promedio de mercado en moneda naci...,% efectivo anual,SBS (web scraping),https://www.sbs.gob.pe/app/pp/EstadisticasSAEE...,1304
4,tasa_referencia_bcrp_pct,Tasa de referencia de la política monetaria de...,% anual,BCRPData (API),PD12301MD,1247
5,tasa_interbancaria_mn_pct,Tasa de interés interbancaria en moneda nacional,% anual,BCRPData (API),PD04692MD,1245
6,riesgo_pais_embig_pbs,Spread EMBIG Perú (riesgo país),puntos básicos,BCRPData (API),PD04709XD,1304
7,tipo_cambio_venta_pen_usd,"Tipo de cambio interbancario, venta",S/ por US$,BCRPData (API),PD04638PD,1246


,id,fecha,tamn_sbs_pct,tipmn_sbs_pct,tasa_referencia_bcrp_pct,tasa_interbancaria_mn_pct,riesgo_pais_embig_pbs,tipo_cambio_venta_pen_usd
0,1,2021-01-01,12.10,0.98,NaN,NaN,132.0,NaN
1,2,2021-01-04,12.23,0.98,0.25,0.25,131.0,3.626667
2,3,2021-01-05,12.15,0.97,0.25,0.25,130.0,3.633667
3,4,2021-01-06,12.10,0.96,0.25,0.25,128.0,3.626833
4,5,2021-01-07,12.08,0.96,0.25,0.25,128.0,3.623000
5,6,2021-01-08,12.05,0.96,0.25,0.25,125.0,3.613833
6,7,2021-01-11,12.06,0.96,0.25,0.25,129.0,3.617333
7,8,2021-01-12,12.11,0.95,0.25,0.25,138.0,3.609000
8,9,2021-01-13,12.10,0.95,0.25,0.25,135.0,3.614000
9,10,2021-01-14,12.10,0.95,0.25,0.25,131.0,3.613500


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>